# NB02 - Agrégation par fenêtre dans Spark Structured Streaming

## A. Mise en place des streams

1. un stream **orders** avec des données de commandes
2. un stream **status** avec les données de mise à jours des commandes

In [0]:
# Passe la variable 'full_path' avec le chemin vers le fichier CSV MOCK_DATA.csv au notebook NB02/Setup
import os

full_path = os.getcwd() + "/Resources/NB02/"

dbutils.widgets.text("full_path", full_path)
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "default")
dbutils.widgets.text("volume", "spark_training")

In [0]:
%run "./Resources/NB02/Setup"

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# schéma pour orders
orders_schema = StructType([
    StructField('customer_id', LongType(), True),
    StructField('notifications', StringType(), True),
    StructField('order_id', LongType(), True),
    StructField('order_timestamp', StringType(), True)
])

# schéma pour la maj des status
status_schema = StructType([
    StructField('order_id', LongType(), True),
    StructField('order_status', StringType(), True),
    StructField('order_timestamp', StringType(), True)
])

# creation du dataframe orders
orders_stream = spark.readStream \
    .format("json") \
    .schema(orders_schema) \
    .option("maxFilesPerTrigger", 1) \
    .option("path", "/Volumes/workspace/default/spark_training/spark_streaming_advanced/data/orders/") \
    .load()

# creation du dataframe status update
status_stream = spark.readStream \
    .format("json") \
    .schema(status_schema) \
    .option("maxFilesPerTrigger", 1) \
    .option("path", "/Volumes/workspace/default/spark_training/spark_streaming_advanced/data/status/") \
    .load()

print(f"orders_stream en cours : {orders_stream.isStreaming}")
print(f"status_stream en cours : {status_stream.isStreaming}")

Faire un notebook qui va copier tout les fichiers dans le catalog

## B. Opérations Stateless vs Stateful
Les opérations **stateless** ne conservent pas d'information sur les données précédemment traitées. Chaque entrée est traitée indépendamment, sans mémoire du passé. Exemples : `select`, `filter`, `map`.

Les opérations **stateful** nécessitent de garder un état entre les traitements, comme l'agrégation sur une fenêtre temporelle ou le suivi d'une clé. Exemples : `groupBy`, `count`, `window`, `dropDuplicates`. Ces opérations sont essentielles pour l'analyse de flux en temps réel, mais demandent plus de ressources pour gérer l'état.

### 1. Example d'opérations Stateless

In [0]:
# suppression du checkpoint (sans ça à la deuxième exécution de la cellule cela cause une erreur)
dbutils.fs.rm("/Volumes/workspace/default/spark_training/spark_streaming_advanced/checkpoint/status", recurse=True)

display(status_stream, streamName="status_stream", checkpointLocation="/Volumes/workspace/default/spark_training/spark_streaming_advanced/checkpoint/status")

In [0]:
# suppression du checkpoint (sans ça à la deuxième exécution de la cellule cela cause une erreur)
dbutils.fs.rm("/Volumes/workspace/default/spark_training/spark_streaming_advanced/checkpoint/orders", recurse=True)

display(orders_stream, streamName="orders_stream", checkpointLocation="/Volumes/workspace/default/spark_training/spark_streaming_advanced/checkpoint/orders")

In [0]:
# on transforme la colonne timestamp vers un format plus lisible
orders_transformed = orders_stream \
    .withColumn("order_time", from_unixtime((col("order_timestamp")) / 1000).cast("timestamp")) \
    .withColumn("notification_enabled", col("notifications") == "Y")


dbutils.fs.rm("/Volumes/workspace/default/spark_training/spark_streaming_advanced/checkpoint/orders_transformed", recurse=True)
display(orders_transformed, streamName="orders_transformed", checkpointLocation="/Volumes/workspace/default/spark_training/spark_streaming_advanced/checkpoint/orders_transformed")

In [0]:
# on fait la même chose pour les status
status_transformed = status_stream \
    .withColumn("status_time", from_unixtime((col("order_timestamp")) / 1000).cast("timestamp")) \


dbutils.fs.rm("/Volumes/workspace/default/spark_training/spark_streaming_advanced/checkpoint/status_transformed", recurse=True)
display(status_transformed, streamName="status_transformed", checkpointLocation="/Volumes/workspace/default/spark_training/spark_streaming_advanced/checkpoint/status_transformed")

%md
### 2. Example d'opérations Stateful

In [0]:
status_counts = status_stream \
    .groupBy("order_status") \
    .count() \
    .orderBy(col("count").desc())

display(status_counts, streamName="status_counts", checkpointLocation="/Volumes/workspace/default/spark_training/spark_streaming_advanced/checkpoint/status_counts/")

## C. Window opérations (fenêtres)

**Attention avec le compute Serverless les cellules suivantes tomberont en erreurs (il faut mettre en place un All-Purpose Compute**

Les opérations de fenêtrage permettent de faire une aggrégation sur une période : 
- Tumbling Windows (Fenêtres à intervalle fixe)
- Sliding Windows

### 1. Exemple Tumbling Windows
On va compter les `orders` par tranche de 1 minute :

In [0]:
# Supprimer les streams qui pourrait avoir le même nom
for query in spark.streams.active:
    if query.name == "tumbling_window_counts":
        query.stop()

# aggrégation
tumbling_windows = status_transformed \
  .groupBy(
    window(col("status_time"), "1 minute"),
    col("order_status")
  ) \
  .count()

# ecriture en mémoire pour viz
tumbling_window_query = tumbling_windows.writeStream \
  .format("memory") \
  .queryName("tumbling_window_counts") \
  .outputMode("complete") \
  .start()

In [0]:
%sql
SELECT 
  window.start as window_start,
  window.end as window_end,
  order_status,
  count
FROM tumbling_window_counts
ORDER BY window_start, order_status

### 2. Exemple Sliding Windows
On va compter le nombre de commandes par tranche de 2 minutes avec un décalage de 1 minute

In [0]:
# Supprimer les streams qui pourrait avoir le même nom
for query in spark.streams.active:
    if query.name == "sliding_window_counts":
        query.stop()

# aggrégation
tumbling_windows = status_transformed \
  .groupBy(
    window(col("status_time"), "2 minutes", "1 minute"),
    col("order_status")
  ) \
  .count()

# ecriture en mémoire pour viz
sliding_window_query = sliding_windows.writeStream \
  .format("memory") \
  .queryName("sliding_window_counts") \
  .outputMode("complete") \
  .start()

In [0]:
%sql
SELECT 
  window.start as window_start,
  window.end as window_end,
  order_status,
  count
FROM sliding_window_counts
ORDER BY window_start, order_status

## D. Jointures de Streams

### 1. Stream-Static JOIN

On va dans un premier temps créer un Dataframe pour réaliser un lookup

In [0]:
# on créer un table static pour le lookup des descriptions des status
status_lookup = spark.createDataFrame([
    (0, "PENDING"),
    (1, "PREPARING"),
    (2, "SHIPPED"),
    (3, "DELIVERED")
], ["order_status", "status_description"])

In [0]:
# on enrichi le stream avec la table de lookup
enriched_status = status_transformed \
    .join(status_lookup, on="order_status")

dbutils.fs.rm("/Volumes/workspace/default/spark_training/spark_streaming_advanced/checkpoint/status_enriched", recurse=True)
display(enriched_status, streamName="enriched_status", checkpointLocation="/Volumes/workspace/default/spark_training/spark_streaming_advanced/checkpoint/status_enriched")


### 2. Jointure Stream-Stream

In [0]:
# Supprimer les streams qui pourrait avoir le même nom
for query in spark.streams.active:
    if query.name == "order_status_join":
        query.stop()

# aggrégation
order_status_join = orders_transformed \
  .join(
    status_transformed, 
    "order_id"
  )

# ecriture en mémoire pour viz
order_status_join_query = order_status_join.writeStream \
  .format("memory") \
  .queryName("order_status_join") \
  .outputMode("append") \
  .start()

In [0]:
%sql
SELECT
  order_id,
  customer_id,
  order_status,
  notifications,
  order_time,
  status_time,
FROM order_status_join
LIMIT 20

## E. Gestion des données tardives avec Watermarks

Watermarks permet de gérer les données tardives et de définir le temps d'attente pour les evênements

In [0]:
for query in spark.streams.active:
    if query.name == "windowed_with_watermark":
        query.stop()

# ajout de watermarks sur status
status_with_watermarks = status_transformed \
    .withWatermark("status_time", "10 minutes")

# fenêter avec watermark
watermarked_windows = status_with_watermark \
    .groupBy(
        window(col("status_time"), "5 minutes"),
        col("order_status")
    ) \
    .count()

# ecriture en mémoire pour viz
watermarked_window_query = watermarked_windows.writeStream \
    .format("memory") \
    .queryName("windowed_with_watermark") \
    .outputMode("complete") \
    .start()

In [0]:
%sql
SELECT 
  window.start as window_start,
  window.end as window_end,
  order_status,
  count
FROM windowed_with_watermark
ORDER BY window_start, order_status